# Module 1: Zenodo SOS Oil-Spill Segmentation on Kaggle

This notebook trains the synopsis Module 1 segmentation core: Sentinel-1 SAR oil-spill masks using DeepLabV3+ / MobileNetV2, BCE+Dice loss, label smoothing, scene-level validation, and optional scSE attention.

**Kaggle feasibility:** yes, Kaggle can download Zenodo archives when Internet is enabled, but the full Zenodo set is about 90 GB compressed before extraction. Use `test_smoke` first to verify the pipeline, then use `module1_balanced` for the real train/validation run. Do not treat smoke-test metrics as final project results.

**Band correction:** the Zenodo SOS TIFFs provide VV/VH Sigma0 data and masks. The full synopsis five-band input (VV, VH, H, alpha, wind-corrected VV/VH) requires your later raw Sentinel-1 + ERA5/CMOD preprocessing. This notebook trains the Kaggle-feasible core on `VV/VH` or `VV/VH + VV-VH dB difference`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, textwrap

# Change REPO_URL to your fork after pushing this branch.
REPO_URL = "https://github.com/reynol-31/Oil_spill_detection.git"
REPO_BRANCH = "codex/kaggle-training"  # use your fork branch; fallback to main if unavailable

# Presets:
# - test_smoke: 9.18 GB compressed; verifies extraction/training only, not final metrics.
# - oil_train: 37.93 GB compressed; positive-class training only, not balanced.
# - module1_balanced: about 80.7 GB compressed; real Module 1 train/val classes.
# - all: about 89.9 GB compressed; includes held-out test archive too.
ARCHIVE_PRESET = "test_smoke"

DOWNLOAD_ARCHIVES = True
EXTRACT_ARCHIVES = True
DELETE_ARCHIVES_AFTER_EXTRACT = False  # set True if disk is tight and extraction succeeded

INPUT_MODE = "vv_vh_diff"  # "vv_vh" or "vv_vh_diff"
USE_SCSE = True             # synopsis v1; set False for v0 baseline
NO_IMAGENET = False         # set True if encoder weight download fails

EPOCHS = 3
BATCH_SIZE = 8
PATCH_SIZE = 256
SAMPLES_PER_SCENE = 4
MAX_SCENES_PER_CLASS = 60   # raise after the first successful run
NUM_WORKERS = 2

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
TEMP = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else WORK / 'temp'
REPO_DIR = WORK / 'Oil_spill_detection'
ARCHIVE_DIR = TEMP / 'zenodo_archives'
EXTRACT_DIR = TEMP / 'sos_extracted'
OUTPUT_DIR = WORK / 'module1_outputs'
for p in [ARCHIVE_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def free_gb(path):
    return shutil.disk_usage(str(path)).free / (1024**3)

print('Python:', sys.version)
print('Working dir:', WORK)
print('Temp dir:', TEMP)
print('Free GB in WORK:', round(free_gb(WORK), 2))
print('Free GB in TEMP:', round(free_gb(TEMP), 2))


## Install Kaggle dependencies

Kaggle already ships a CUDA-matched PyTorch build. This cell installs the missing Python packages without reinstalling `torch`.


In [ ]:
packages = [
    'segmentation-models-pytorch',
    'tifffile',
    'py7zr',
    'tqdm',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Clone the project code

If you pushed this work to a fork, set `REPO_URL` to that fork. The original repository does not contain this branch until it is pushed.


In [ ]:
if not (REPO_DIR / 'src').exists():
    try:
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    except subprocess.CalledProcessError:
        print(f'Branch {REPO_BRANCH!r} was not available at {REPO_URL}; cloning main instead.')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repo ready at', REPO_DIR)


## Zenodo archive plan

Your direct Part I link is valid, but it is only the oil-image archive. A balanced segmentation model also needs masks plus no-oil/look-alike examples from Part II. The notebook uses Zenodo API `content` URLs because they are stable for scripted download.


In [ ]:
from src.training.zenodo_sos_dataset import ZENODO_ARCHIVES, download_zenodo_archives, extract_7z, discover_sos_pairs

PRESETS = {
    'test_smoke': ['test_all'],
    'oil_train': ['oil_images', 'oil_masks'],
    'module1_balanced': [
        'oil_images', 'oil_masks',
        'no_oil_images', 'no_oil_masks',
        'lookalike_images', 'lookalike_masks',
    ],
    'all': list(ZENODO_ARCHIVES.keys()),
}
archive_keys = PRESETS[ARCHIVE_PRESET]
print('Selected preset:', ARCHIVE_PRESET)
print('Archives:')
for key in archive_keys:
    spec = ZENODO_ARCHIVES[key]
    print(f"  {key:16s} {spec['size_gb']:6.2f} GB  {spec['filename']}")
print('Estimated compressed GB:', round(sum(ZENODO_ARCHIVES[k]['size_gb'] for k in archive_keys), 2))

if ARCHIVE_PRESET in {'module1_balanced', 'all'}:
    print('
Large run warning: this can exceed a free Kaggle session after extraction. Consider reducing MAX_SCENES_PER_CLASS or preparing a Kaggle Dataset with already-extracted TIFFs.')


In [ ]:
if DOWNLOAD_ARCHIVES:
    archives = download_zenodo_archives(archive_keys, ARCHIVE_DIR, min_free_gb_after_download=8.0)
else:
    archives = [ARCHIVE_DIR / ZENODO_ARCHIVES[k]['filename'] for k in archive_keys]

if EXTRACT_ARCHIVES:
    for archive in archives:
        marker = EXTRACT_DIR / (archive.stem + '.extracted.ok')
        if marker.exists():
            print('Already extracted:', archive.name)
            continue
        print('Extracting:', archive.name)
        extract_7z(archive, EXTRACT_DIR)
        marker.write_text('ok')
        if DELETE_ARCHIVES_AFTER_EXTRACT:
            archive.unlink(missing_ok=True)

print('Free GB in TEMP after data step:', round(free_gb(TEMP), 2))


## Discover TIFF image/mask pairs

The split is scene-level, not random patch-level, to avoid leakage between neighboring patches from the same SAR scene.


In [ ]:
df = discover_sos_pairs(EXTRACT_DIR)
print(df.head())
print('
Class counts:')
print(df['class_name'].value_counts(dropna=False))
print('
Mask availability:')
print(df.groupby('class_name')['has_mask'].mean())

if df.empty:
    raise RuntimeError('No TIFF files were found. Check EXTRACT_DIR and archive extraction logs.')


## Train DeepLabV3+ Module 1

This calls the repo training entrypoint. Outputs are saved under `/kaggle/working/module1_outputs` so Kaggle can persist the best checkpoint and metrics.


In [ ]:
cmd = [
    sys.executable, '-m', 'src.training.kaggle_module1_train',
    '--data-root', str(EXTRACT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--patch-size', str(PATCH_SIZE),
    '--samples-per-scene', str(SAMPLES_PER_SCENE),
    '--max-scenes-per-class', str(MAX_SCENES_PER_CLASS),
    '--num-workers', str(NUM_WORKERS),
    '--input-mode', INPUT_MODE,
]
if USE_SCSE:
    cmd.append('--use-scse')
if NO_IMAGENET:
    cmd.append('--no-imagenet')

print('Running:')
print(' '.join(cmd))
subprocess.check_call(cmd)


## Review metrics

For `test_smoke`, use this only to confirm the pipeline runs. For final reporting, use `module1_balanced` or a proper train/validation dataset and keep Part III as external test data.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = OUTPUT_DIR / 'metrics.csv'
metrics = pd.read_csv(metrics_path)
display(metrics)

ax = metrics.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', figsize=(7, 4), title='Loss')
ax.grid(True)
plt.show()

ax = metrics.plot(x='epoch', y=['train_miou', 'val_miou', 'train_f1', 'val_f1'], marker='o', figsize=(7, 4), title='Segmentation metrics')
ax.grid(True)
plt.show()


## Visual sanity check

This loads one validation patch, predicts a mask, and displays input/prediction/ground truth side by side.


In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from src.training.zenodo_sos_dataset import PatchConfig, SOSTiffPatchDataset
from src.training.kaggle_module1_train import build_model

val_df = pd.read_csv(OUTPUT_DIR / 'val_scenes.csv')
run_config = json.loads((OUTPUT_DIR / 'run_config.json').read_text())
patch_config = PatchConfig(
    patch_size=run_config['patch_size'],
    input_mode=run_config['input_mode'],
    samples_per_scene=1,
    seed=run_config['seed'],
    augment=False,
)
val_ds = SOSTiffPatchDataset(val_df, patch_config, train=False)
sample = val_ds[0]
ckpt = torch.load(OUTPUT_DIR / 'best_model.pt', map_location='cpu')
model = build_model(
    in_channels=run_config['in_channels'],
    use_scse=run_config['use_scse'],
    patch_size=run_config['patch_size'],
    encoder_weights=None,
)
model.load_state_dict(ckpt['model'])
model.eval()
with torch.no_grad():
    logits = model(sample['image'][None])
    pred = torch.sigmoid(logits)[0, 0].numpy()

img = sample['image'].numpy()
mask = sample['mask'][0].numpy()
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(img[0], cmap='gray'); axes[0].set_title('VV')
axes[1].imshow(img[1], cmap='gray'); axes[1].set_title('VH')
axes[2].imshow(pred > 0.5, cmap='magma'); axes[2].set_title('Prediction')
axes[3].imshow(mask, cmap='magma'); axes[3].set_title('Ground truth')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()


## Next steps toward the full synopsis

1. Use `module1_balanced` for the real Module 1 train/validation run.
2. Keep Part III as a held-out test set once you have trained on Part I/II.
3. Add the raw Sentinel-1 preprocessing pipeline for H/alpha and wind-corrected ratio bands before claiming the full five-band model.
4. Train Module 2 Random Forest look-alike rejection from connected components produced by this model.
5. Integrate AIS, drift attribution, and confidence scoring after segmentation is stable.
